<a href="https://colab.research.google.com/github/heytian/d2d-oco3-tools/blob/main/20260404_02_updateDuckDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Update DuckDB (OCO-3 CO2 & SIF) based on Clasp Report 356 SAMs**


- 20260404 Step 2 to "20260404_01_centroids+ne.py" Step 1 in same git repo.
- Background: Step 1 was to update Clasp Report (centroid location information from JPL team on SAM targets) with Natural Earth population and city name information
- **IMPORTANT**: Clear all outputs and end runtime session before saving to Colab or Github to avoid exposing your Earthdata credentials!


**Development notes (Work in progress as of Apr 4, 2026): this is with reference to the NetCDF to DuckDB pipeline from March 2026 named "nc4-timezone-pop.ipynb" in the same github repo.



In [3]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [1]:
import duckdb
import geopandas as gpd
import os, shutil
import pandas as pd
from shapely import wkt

con = "/content/drive/MyDrive/Shortcuts/columnar/oco3_data.duckdb"


In [2]:
# check DuckDB data structure
con.execute("SHOW TABLES").df()

AttributeError: 'str' object has no attribute 'execute'

In [ ]:
# TO CHECK THROUGH

# !pip install duckdb geopandas shapely --quiet

DB_SRC    = "/content/drive/MyDrive/Shortcuts/columnar/oco3_data.duckdb"
DB_LOCAL  = "/content/oco3_data.duckdb"
WKT_PATH  = "/content/drive/MyDrive/Shortcuts/csv_xlsx/20260129_from_Rob/clasp_report.csv"

PARQUET_DIR = "/content/drive/MyDrive/Shortcuts/columnar/oco3_parquet"
CSV_DIR     = "/content/drive/MyDrive/Shortcuts/csv_xlsx/from-duckdb"

TABLES = {
    "co2_sam": "xco2",
    "sif_sam":  "Daily_SIF_757nm",
}

# ── 1. Load updated clasp_report ───────────────
print("Loading updated clasp_report...")
ref_data    = pd.read_csv(WKT_PATH)
ref_geodata = gpd.GeoDataFrame(
    ref_data,
    geometry=ref_data["Site Shape WKT"].apply(wkt.loads),
    crs="EPSG:4326"
)

# Lookup: target_name → City, Country, Population
target_meta = ref_data.set_index("Target Name")[["City", "Country", "Population"]]

print(f"  {len(ref_geodata)} targets loaded")
print(f"  Sample:\n{target_meta.head(3)}")

def spatial_filter(df):
    pts = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.longitude, df.latitude),
        crs="EPSG:4326"
    )
    joined = gpd.sjoin(
        pts,
        ref_geodata[["Target Name", "geometry"]],
        how="inner", predicate="within"
    )
    if joined.empty:
        return None
    joined = joined.rename(columns={"Target Name": "target_name"})
    joined = joined.drop(columns=["geometry", "index_right"], errors="ignore")

    joined = joined.join(target_meta, on="target_name", how="left")

    return joined

print("\nCopying DB locally...")
shutil.copy(DB_SRC, DB_LOCAL)
print(f"  Local DB: {os.path.getsize(DB_LOCAL)/1024**2:.1f} MB")

con = duckdb.connect(DB_LOCAL)

for table in TABLES:
    print(f"\n{'='*50}")
    print(f"Processing: {table}")
    print(f"{'='*50}")

    try:
        total_before = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"  Rows before: {total_before:,}")
    except Exception as e:
        print(f"  Table {table} not found: {e}, skipping")
        continue

    df = con.execute(f"SELECT * FROM {table}").df()
    print(f"  Loaded {len(df):,} rows")

    print("  Running spatial filter...")
    filtered = spatial_filter(df)

    if filtered is None or filtered.empty:
        print(f"  WARNING: No rows survived spatial filter for {table}!")
        continue

    print(f"  Rows after filter: {len(filtered):,} (dropped {len(df)-len(filtered):,})")

    original_cols = list(con.execute(f"DESCRIBE {table}").df()['column_name'])
    final_cols    = [c for c in original_cols if c in filtered.columns]
    for col in ['City', 'Country', 'Population']:
        if col not in final_cols and col in filtered.columns:
            final_cols.append(col)

    filtered = filtered[final_cols]

    print(f"  Truncating and reloading {table}...")
    con.execute(f"DELETE FROM {table}")
    con.execute(f"INSERT INTO {table} SELECT * FROM filtered")

    total_after = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  Rows after reload: {total_after:,}")

con.close()

print("\n── Verification ──────────────────────────")
con = duckdb.connect(DB_LOCAL)
for table in TABLES:
    try:
        row = con.execute(f"""
            SELECT COUNT(*)                    AS rows,
                   COUNT(DISTINCT target_name) AS targets,
                   COUNT(DISTINCT City)        AS cities,
                   MIN(datetime)               AS earliest,
                   MAX(datetime)               AS latest
            FROM {table}
        """).fetchone()
        print(f"{table}: {row[0]:,} rows | {row[1]} targets | {row[2]} cities | {row[3]} → {row[4]}")
    except Exception as e:
        print(f"{table}: {e}")
con.close()

print("\nCopying back to Drive...")
shutil.copy(DB_LOCAL, DB_SRC)
print(f"  Drive DB: {os.path.getsize(DB_SRC)/1024**2:.1f} MB")

os.makedirs(PARQUET_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

con = duckdb.connect(DB_SRC)
for table in TABLES:
    parquet_path = f"{PARQUET_DIR}/{table}.parquet"
    csv_path     = f"{CSV_DIR}/{table}.csv"

    con.execute(f"COPY (SELECT * FROM {table}) TO '{parquet_path}' (FORMAT PARQUET)")
    print(f"  Parquet: {table} → {os.path.getsize(parquet_path)/1024**2:.1f} MB")

    con.execute(f"COPY {table} TO '{csv_path}' (HEADER, DELIMITER ',')")
    print(f"  CSV:     {table} → {os.path.getsize(csv_path)/1024**2:.1f} MB")

con.close()
print("\nAll done.")